## It updates Python’s path to allow importing project modules from parent directories.

In [0]:
import os
import sys
project_path=(os.path.join(os.getcwd(),'..','..'))
sys.path.append(project_path)

from utils.transformation import reusable

##object creation

In [0]:
df_user_obj=reusable()

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

### **DimUser**

In [0]:
# df=spark.read.format("parquet").load("abfss://bornze@storagespotifyproject20.dfs.core.windows.net/DimUser")

## Auto Loader

In [0]:
df=spark.readStream.format('cloudFiles').option("cloudFiles.format", "parquet").\
  option("cloudFiles.schemaLocation","abfss://silver@storagespotifyproject20.dfs.core.windows.net/DimUser/schema")\
    .load("abfss://bornze@storagespotifyproject20.dfs.core.windows.net/DimUser")

## Changing user_name lower case to upper_case

In [0]:
from pyspark.sql.functions import *
df_user=df.withColumn('user_name',upper(col('user_name')))

## Creating object for my transformation function

In [0]:
df_user_obj=reusable()
df_user=df_user_obj.dropColumns(df_user,['_rescued_data'])
df_user=df_user_obj.dropDuplicates(df_user,['user_id'])
# display(df_user)

## Write the Data in the form of delta trigger=once meanse once the data go processed it got stopped

In [0]:
df_user.writeStream.format("delta").\
    outputMode("append").option("checkpointLocation","abfss://silver@storagespotifyproject20.dfs.core.windows.net/DimUser/checkpoint").\
        trigger(once=True).\
            option("path","abfss://silver@storagespotifyproject20.dfs.core.windows.net/DimUser/data").\
                toTable("spotify_cata.silver.DimUser")

##DimArtist

## in read this is not a checkpoint : option("cloudFiles.schemaLocation","pathname").here we are cheking the cloudFiles.schemaLocation.means any Store inferred schema,Track schema evolution

In [0]:
df_artist=spark.readStream.format('cloudFiles').\
    option("cloudFiles.format", "parquet").\
  option("cloudFiles.schemaLocation","abfss://silver@storagespotifyproject20.dfs.core.windows.net/DimArt/schema")\
    .load("abfss://bornze@storagespotifyproject20.dfs.core.windows.net/DimArtist")

##Transformation

In [0]:
df_user_obj=reusable()
df_artist=df_user_obj.dropColumns(df_artist,['_rescued_data'])
df_artist=df_user_obj.dropDuplicates(df_artist,['artist_id'])
# display(df_artist)

##Write

In [0]:
df_artist.writeStream.format("delta").\
    outputMode("append").option("checkpointLocation","abfss://silver@storagespotifyproject20.dfs.core.windows.net/DimArt/checkpoint").\
        trigger(once=True).option("path","abfss://silver@storagespotifyproject20.dfs.core.windows.net/DimArt/data").\
            toTable("spotify_cata.silver.DimArt")

##DimTrack

In [0]:
df_Track=spark.readStream.format('cloudFiles').option("cloudFiles.format", "parquet").\
  option("cloudFiles.schemaLocation","abfss://silver@storagespotifyproject20.dfs.core.windows.net/DimTrack/schema")\
    .load("abfss://bornze@storagespotifyproject20.dfs.core.windows.net/DimTrack")

In [0]:
df_Track = df_Track.withColumn(
    'duration_flag',
    when(col('duration_sec') <= 150, 'short')
    .when((col('duration_sec') > 150) & (col('duration_sec') <= 300), 'medium')
    .when(col('duration_sec') > 300, 'High')
    .otherwise('Extreme High')
).withColumn('track_name',regexp_replace(col('track_name'),'-',' '))

df_Track=df_user_obj.dropColumns(df_Track,['_rescued_data'])


##Write

In [0]:
df_Track.writeStream.format("delta").\
    outputMode("append").option("checkpointLocation","abfss://silver@storagespotifyproject20.dfs.core.windows.net/DimTrack/checkpoint").\
        trigger(once=True).option("path","abfss://silver@storagespotifyproject20.dfs.core.windows.net/DimTrack/data").\
            toTable("spotify_cata.silver.DimTrack")

##DimDate

In [0]:
df_date=spark.readStream.format('cloudFiles').option("cloudFiles.format", "parquet").\
  option("cloudFiles.schemaLocation","abfss://silver@storagespotifyproject20.dfs.core.windows.net/DimDate/schema")\
    .load("abfss://bornze@storagespotifyproject20.dfs.core.windows.net/DimDate")

In [0]:
df_user_obj=reusable()
df_date=df_user_obj.dropColumns(df_date,['_rescued_data'])

##Write

In [0]:
df_date.writeStream.format("delta").\
    outputMode("append").option("checkpointLocation","abfss://silver@storagespotifyproject20.dfs.core.windows.net/DimDate/checkpoint").\
        trigger(once=True).option("path","abfss://silver@storagespotifyproject20.dfs.core.windows.net/DimDate/data").\
            toTable("spotify_cata.silver.DimDate")

##FactStream

In [0]:
df_FactStream=spark.readStream.format('cloudFiles').option("cloudFiles.format", "parquet").\
  option("cloudFiles.schemaLocation","abfss://silver@storagespotifyproject20.dfs.core.windows.net/FactStream/schema")\
    .load("abfss://bornze@storagespotifyproject20.dfs.core.windows.net/FactStream")

In [0]:
df_user_obj=reusable()
df_FactStream=df_user_obj.dropColumns(df_FactStream,['_rescued_data'])

In [0]:
df_FactStream.writeStream.format("delta").\
    outputMode("append").option("checkpointLocation","abfss://silver@storagespotifyproject20.dfs.core.windows.net/FactStream/checkpoint").\
        trigger(once=True).option("path","abfss://silver@storagespotifyproject20.dfs.core.windows.net/FactStream/data").\
            toTable("spotify_cata.silver.FactStream")

In [0]:
%sql
-- select * from spotify_cata.gold.dimtrack  where 
-- track_id in(46,5)
-- --`__END_AT` is not null